# Conditional training on a synthetic Gaussian family

This example covers the full training path. You build a conditional model,
draw target bitstrings from a synthetic family, train the parameters, sample
the trained model through the PennyLane bridge, and score the samples against
the targets.

A conditional IQP model applies one shared diagonal unitary to a
condition-dependent input state. The condition changes only the input state, so
the learned parameters are shared across every condition. Training estimates a
multiscale Hamming-RBF maximum mean discrepancy from Pauli-character samples,
which stays classical whenever the encoder implements the interface described in
the README.

The target is a translated discrete Gaussian over Hamming weight. Condition `c`
sets the mean weight to `n * (0.15 + 0.70 * c)` with a fixed width of
`0.5 * sqrt(n)`. The encoder is a product state with one angle per condition,
`alpha(c) = pi * (0.35 - 0.70 * c)`, so most of the distribution has to be
learned from the data.

The register holds 12 qubits, which keeps the exact sampler and the PennyLane
bridge cheap, and every cell here runs in seconds.


## Setup

Run the notebook from a checkout of the repository after `uv sync`, which
installs the library in editable mode together with the notebook tooling. Run
the cells in order, because later cells use `training`, `model`, and `result`
from earlier ones.

Importing `iqp_cond` turns on JAX float64. The estimator, the loss, and the
trainer depend on that setting.

The budgets in the import cell are smaller than the library defaults so the
notebook finishes in seconds.


In [1]:
import numpy as np

import iqp_cond as ciqp

N_QUBITS = 12
TRAINING_CONDITIONS = (0.0, 0.25, 0.5, 0.75, 1.0)
N_TRAIN, N_VALIDATION = 500, 200

# The library defaults are 2000 updates with 512 masks and 1000 Monte Carlo
# samples per update. This run is short so the notebook stays interactive.
STEPS, BATCH_SIZE, N_OPS, MC_SAMPLES = 500, 100, 128, 2048

## Build the target family

Targets are bitstrings shaped `(conditions, samples, qubits)`. `gaussian_samples`
draws a Hamming weight from the Gaussian law and then places that many ones at
uniformly random positions. Only the weight law depends on the condition, so
each condition is exchangeable in the qubit index. Matching the weight law is
necessary but not sufficient, since the positions carry structure as well, and
that is what the learned diagonal has to reproduce.


In [2]:
def gaussian_samples(
    rng: np.random.Generator,
    condition: float,
    n_samples: int,
) -> np.ndarray:
    """Bitstrings whose Hamming weight follows the Gaussian law."""
    # The condition moves the mean weight and leaves the width alone.
    mean = N_QUBITS * (0.15 + 0.70 * condition)
    width = 0.5 * np.sqrt(N_QUBITS)
    weights = np.arange(N_QUBITS + 1)
    shell_probabilities = np.exp(-0.5 * ((weights - mean) / width) ** 2)
    shell_probabilities /= shell_probabilities.sum()
    drawn_weights = rng.choice(N_QUBITS + 1, size=n_samples, p=shell_probabilities)
    samples = np.zeros((n_samples, N_QUBITS))
    # Place the drawn number of ones at positions sampled without replacement.
    for row, weight in enumerate(drawn_weights):
        if weight:
            samples[row, rng.choice(N_QUBITS, size=int(weight), replace=False)] = 1.0
    return samples

`hamming_profile` collapses a bitstring sample into the probability of each
Hamming weight, which is what the final score compares. Hamming weight is the
natural summary here because the target law is written in terms of it.


In [3]:
def hamming_profile(bits: np.ndarray) -> np.ndarray:
    """Probability of each Hamming weight under a bitstring sample."""
    weights = np.asarray(bits).sum(axis=-1).astype(int)
    # bincount counts how many samples land on each possible weight.
    counts = np.bincount(weights, minlength=N_QUBITS + 1)
    return counts / weights.size

## Draw the data

Training and validation targets come from independent draws of the same law. The
trainer selects the checkpoint with the best validation loss, so keep the
validation split out of anything you report later.


In [4]:
rng = np.random.default_rng(20260917)
conditions = np.asarray(TRAINING_CONDITIONS)
# One leading axis for the conditions, then samples, then qubits.
training = np.stack([gaussian_samples(rng, c, N_TRAIN) for c in conditions])
validation = np.stack([gaussian_samples(rng, c, N_VALIDATION) for c in conditions])

print(f"training targets {training.shape}, validation targets {validation.shape}")
print(f"mean Hamming weight per condition {training.sum(axis=-1).mean(axis=-1)}")

training targets (5, 500, 12), validation targets (5, 200, 12)
mean Hamming weight per condition [2.16  3.872 6.096 7.938 9.87 ]


## Build the model

A `ConditionalModel` pairs a shared generator bank with an encoder.
`Ansatz.complete(n)` builds the bank from one- and two-body generators, and the
trainable diagonal is `exp(i * sum_a theta_a Z^a)` over those generators. The
encoder supplies the condition dependence before the first Hadamard layer, so
the generator bank stays shared.

`AngleEncoder` takes angles shaped `(conditions, layers, qubits)` and one axis
label per layer and qubit. A single all-`Y` layer is the plain `RY` product
state, which is what this example uses.


In [5]:
# One angle per condition, alpha(c) = pi * (0.35 - 0.70 * c), the same on every
# qubit.
angles = np.repeat((np.pi * (0.35 - 0.70 * conditions))[:, None], N_QUBITS, axis=1)
model = ciqp.ConditionalModel(
    ciqp.Ansatz.complete(N_QUBITS),
    # (conditions, layers, qubits), one layer of Y rotations.
    ciqp.AngleEncoder(angles[:, None, :], axes=("Y" * N_QUBITS,)),
)
print(f"{model.n_qubits} qubits, {conditions.size} conditions")

12 qubits, 5 conditions


## Train

`train` returns a `TrainingResult`. Each update samples one condition and one
bandwidth from the configured objective cells and averages the Monte Carlo
samples inside that cell, so the loss and its gradient stay unbiased for the
multiscale objective. Validation evaluates every cell with one fixed operator
bank, which keeps `best_validation_loss` comparable between steps.

Parameters start from the pooled training conditions under the mean-covariance
prescription, which matches the one- and two-body structure of the data instead
of starting from noise. A run this short stops before the validation loss
flattens.


In [6]:
result = ciqp.train(
    model,
    training,
    validation,
    seed=0,
    config=ciqp.TrainingConfig(
        steps=STEPS,
        batch_size=BATCH_SIZE,
        # Nonidentity Fourier masks per update, and Monte Carlo reference draws
        # averaged for each mask.
        n_ops=N_OPS,
        mc_samples=MC_SAMPLES,
    ),
)
print(f"initial validation MMD {result.validation_losses[0]:.6f}")
print(f"best step {result.best_step}")
print(f"validation MMD {result.best_validation_loss:.6f}")

initial validation MMD 0.083858
best step 500
validation MMD 0.035100


## Sample the trained parameters

Training never uses PennyLane. Sampling compiles the trained model into a
PennyLane circuit: the encoder preparation, one Hadamard layer, the shared
diagonal as one `MultiRZ` per generator, and a second Hadamard layer.

`model.sample` draws shots and accepts any PennyLane device, including hardware
backends. `model.probabilities` returns the exact visible distribution, which
needs an analytic device and a register that fits in memory. This register fits.


In [7]:
# Shot samples from the compiled circuit, shaped (conditions, samples, qubits).
draws = model.sample(result.best_params, 2000, seed=1)
# Exact statevector probabilities, shaped (conditions, 2 ** qubits).
probabilities = model.probabilities(result.best_params)
print(f"shot draws {draws.shape}")
print(f"exact probabilities {probabilities.shape}")

shot draws (5, 2000, 12)
exact probabilities (5, 4096)


## Score the result

Marginalizing the exact distribution onto Hamming weight gives a profile
directly comparable with the target sample. The target profile is itself an
estimate from `N_TRAIN` draws, so part of that distance is target sampling noise
rather than model error. Raise `N_TRAIN` before reading small differences as
real.


In [8]:
# Hamming weight of every computational basis state, indexed by state integer.
state_weights = np.bitwise_count(np.arange(2**N_QUBITS)).astype(int)
profile_tvds = []
for index in range(conditions.size):
    model_profile = np.bincount(
        state_weights,
        weights=probabilities[index],
        minlength=N_QUBITS + 1,
    )
    target_profile = hamming_profile(training[index])
    # Total variation distance between the two weight distributions.
    profile_tvds.append(0.5 * float(np.abs(model_profile - target_profile).sum()))

print(f"per-condition profile TVD {np.round(profile_tvds, 3)}")
print(f"mean profile TVD {np.mean(profile_tvds):.3f}")

per-condition profile TVD [0.513 0.422 0.086 0.399 0.532]
mean profile TVD 0.390


## Where to go next

Raise `STEPS`, `N_OPS`, and `MC_SAMPLES` to tighten the fit. The
`TrainingConfig` docstring records the full-scale budget of 2000 masks and 8192
reference draws, close to what reported numbers use.

The same objects support more than the training loop. `model.mmd` scores a
parameter vector against new targets without training,
`model.estimate_pauli_z` returns moment estimates for masks you choose, and
`examples/custom_encoder.ipynb` replaces the built-in encoder with one you write
yourself.
